# ⚛️ Módulo 5: Dinámica Molecular
## Actividad 5.4: Termostatos y Barostatos – Simulaciones NVE, NVT, NPT

<div align="center">
  
**Universidad de Caldas - Departamento de Química**  
*Introducción a la Química Computacional (173G7G)*  
**Profesor:** José Mauricio Rodas Rodríguez

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/maurorodas/Quimica_computacional_173G7G/blob/main/modulo_05_dinamica_molecular/04_termostatos_barostatos.ipynb)

</div>

---

## 🎯 Objetivos de Aprendizaje

Al finalizar esta actividad, serás capaz de:
- Configurar correctamente los parámetros de termostato y barostato en archivos `.mdp` de GROMACS
- Comparar los termostatos más utilizados: Berendsen, v-rescale y Nosé-Hoover
- Implementar y analizar simulaciones NVE, NVT y NPT con OpenMM
- Monitorear la equilibración del sistema mediante propiedades termodinámicas
- Validar que el sistema ha alcanzado equilibrio antes de la producción

---

## 1. Instalación de Dependencias

In [ ]:
# Para OpenMM instalar con conda es más confiable:
# conda install -c conda-forge openmm
!pip install numpy matplotlib scipy
# OpenMM se instala opcionalmente
try:
    import openmm
    print("OpenMM disponible:", openmm.__version__)
except ImportError:
    print("OpenMM no disponible. Instala con: conda install -c conda-forge openmm")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.constants import k as k_B

print("Bibliotecas importadas correctamente")

## 2. Termostatos: Comparación Detallada

### 2.1 Termostato de Berendsen

Reescala velocidades con una constante de tiempo $\tau_T$:

$$\lambda = \sqrt{1 + \frac{\Delta t}{\tau_T}\left(\frac{T_0}{T} - 1\right)}$$

**Pros:** Converge rápido a la temperatura objetivo.  
**Contras:** No genera el ensamble NVT correcto; suprime las fluctuaciones de energía.

### 2.2 Termostato v-rescale (GROMACS por defecto)

Como Berendsen pero con un término estocástico que corrige las fluctuaciones:

$$dK = (K_0 - K)\frac{dt}{\tau_T} + 2\sqrt{\frac{K K_0}{N_f}}\frac{dW}{\sqrt{\tau_T}}$$

**Pros:** Genera el ensamble NVT correcto. Es el recomendado para la mayoría de simulaciones.

### 2.3 Termostato de Nosé-Hoover

Extiende las ecuaciones de movimiento con una variable de baño térmica $\xi$:

$$\dot{\mathbf{p}}_i = \mathbf{F}_i - \xi \mathbf{p}_i, \qquad \dot{\xi} = \frac{1}{Q}\left(\sum_i \frac{p_i^2}{m_i} - 3Nk_BT\right)$$

**Pros:** Genera el ensamble NVT exacto; conserva una hamiltoniana extendida.  
**Contras:** Puede oscilar si $Q$ (masa del baño) no está bien elegido.

In [ ]:
# Simulación comparativa de termostatos en un gas ideal
np.random.seed(42)

N = 200           # partículas
T_objetivo = 300  # K
T_inicial = 500   # K (comenzamos calientes)
masa = 39.948 * 1.66054e-27  # kg (Ar)
dt = 1e-14        # 10 fs (exagerado para ver efecto)
tau_T = 200 * dt  # tiempo de acoplamiento
n_pasos = 600

def T_instantanea(vel, m, N):
    return np.sum(m * np.sum(vel**2, axis=1)) / (3 * N * k_B)

def berendsen_step(vel, m, N, T_obj, dt, tau):
    T = T_instantanea(vel, m, N)
    lam = np.sqrt(1 + dt / tau * (T_obj / T - 1))
    return vel * lam

def vrescale_step(vel, m, N, T_obj, dt, tau, rng):
    T = T_instantanea(vel, m, N)
    Nf = 3 * N  # grados de libertad
    K = 0.5 * np.sum(m * np.sum(vel**2, axis=1))
    K0 = 0.5 * Nf * k_B * T_obj
    # término determinista + estocástico
    dK = (K0 - K) * dt / tau + 2 * np.sqrt(K * K0 / Nf) * rng.normal() * np.sqrt(dt / tau)
    lam = np.sqrt((K + dK) / K) if K > 0 else 1.0
    return vel * lam

sigma_ini = np.sqrt(k_B * T_inicial / masa)
vel0 = np.random.normal(0, sigma_ini, (N, 3))
vel0 -= vel0.mean(axis=0)
masas_arr = np.full(N, masa)

vel_B  = vel0.copy()
vel_vr = vel0.copy()
rng = np.random.default_rng(0)

T_B  = [T_instantanea(vel_B,  masa, N)]
T_vr = [T_instantanea(vel_vr, masa, N)]

for _ in range(n_pasos):
    # Pequeña perturbación dinámica
    dv = np.random.normal(0, sigma_ini * 0.005, (N, 3))
    vel_B  += dv
    vel_vr += dv.copy()

    vel_B  = berendsen_step(vel_B,  masa, N, T_objetivo, dt, tau_T)
    vel_vr = vrescale_step( vel_vr, masa, N, T_objetivo, dt, tau_T, rng)

    T_B.append(T_instantanea(vel_B,  masa, N))
    T_vr.append(T_instantanea(vel_vr, masa, N))

pasos = np.arange(n_pasos + 1)
plt.figure(figsize=(10, 5))
plt.plot(pasos, T_B,  'b-',  label='Berendsen',  linewidth=1.5, alpha=0.8)
plt.plot(pasos, T_vr, 'g-',  label='v-rescale',  linewidth=1.5, alpha=0.8)
plt.axhline(T_objetivo, color='r', linestyle='--', linewidth=2, label=f'T = {T_objetivo} K')
plt.xlabel('Paso', fontsize=12)
plt.ylabel('Temperatura (K)', fontsize=12)
plt.title('Comparación de Termostatos: Berendsen vs v-rescale', fontsize=13)
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('comparacion_termostatos.png', dpi=100, bbox_inches='tight')
plt.show()

equilibrio = slice(300, None)
print(f"Berendsen  - T media (equilibrio): {np.mean(T_B[300:]):.2f} K, σ = {np.std(T_B[300:]):.2f} K")
print(f"v-rescale  - T media (equilibrio): {np.mean(T_vr[300:]):.2f} K, σ = {np.std(T_vr[300:]):.2f} K")

## 3. Archivos de Configuración GROMACS (.mdp)

En GROMACS, todos los parámetros de la simulación se especifican en un archivo `.mdp` (Molecular Dynamics Parameters).

In [ ]:
# Generar archivos .mdp típicos para cada etapa de simulación

mdp_minimizacion = """\
; Archivo MDP: Minimización de Energía
; =====================================
integrator    = steep        ; Steepest descent
emtol         = 1000.0       ; Convergencia (kJ/mol/nm)
emstep        = 0.01         ; Paso inicial
nsteps        = 50000        ; Máximo de pasos

; Electrostática y VdW
cutoff-scheme = Verlet
coulombtype   = PME
rcoulomb      = 1.0          ; Radio de corte (nm)
rvdw          = 1.0          ; Radio de corte VdW (nm)
pbc           = xyz          ; Condiciones periódicas
"""

mdp_nvt = """\
; Archivo MDP: Equilibración NVT (100 ps)
; =========================================
integrator    = md           ; Leap-Frog
dt            = 0.002        ; Paso de tiempo (ps)
nsteps        = 50000        ; 100 ps total
nstxout       = 500          ; Guardar coordenadas cada 1 ps
nstvout       = 500          ; Guardar velocidades cada 1 ps
nstenergy     = 500          ; Guardar energías cada 1 ps

; Termostato
tcoupl        = V-rescale    ; Termostato v-rescale
tc-grps       = Protein Non-Protein
tau_t         = 0.1  0.1     ; Constantes de acoplamiento (ps)
ref_t         = 300  300     ; Temperatura objetivo (K)

; Sin barostato en NVT
pcoupl        = no

; Electrostática y VdW
cutoff-scheme = Verlet
coulombtype   = PME
rcoulomb      = 1.0
rvdw          = 1.0
pbc           = xyz

; Restricciones
constraints   = h-bonds      ; Restringir enlaces con H
constraint-algorithm = LINCS

; Restricción de posiciones (átomos pesados de la proteína)
define        = -DPOSRES
"""

mdp_npt = """\
; Archivo MDP: Equilibración NPT (100 ps)
; =========================================
integrator    = md
dt            = 0.002
nsteps        = 50000        ; 100 ps
nstxout       = 500
nstvout       = 500
nstenergy     = 500

; Termostato
tcoupl        = V-rescale
tc-grps       = Protein Non-Protein
tau_t         = 0.1  0.1
ref_t         = 300  300

; Barostato Parrinello-Rahman
pcoupl        = Parrinello-Rahman
pcoupltype    = isotropic    ; Compresión isótropa
tau_p         = 2.0          ; Constante de acoplamiento (ps)
ref_p         = 1.0          ; Presión objetivo (bar)
compressibility = 4.5e-5     ; Compresibilidad del agua (bar^-1)

; Electrostática y VdW
cutoff-scheme = Verlet
coulombtype   = PME
rcoulomb      = 1.0
rvdw          = 1.0
pbc           = xyz

; Restricciones
constraints   = h-bonds
constraint-algorithm = LINCS
"""

mdp_produccion = """\
; Archivo MDP: Producción NPT (10 ns)
; =====================================
integrator    = md
dt            = 0.002
nsteps        = 5000000      ; 10 ns total
nstxout       = 5000         ; Guardar cada 10 ps
nstvout       = 5000
nstenergy     = 5000
nstlog        = 5000

; Termostato
tcoupl        = V-rescale
tc-grps       = Protein Non-Protein
tau_t         = 0.1  0.1
ref_t         = 300  300

; Barostato
pcoupl        = Parrinello-Rahman
pcoupltype    = isotropic
tau_p         = 2.0
ref_p         = 1.0
compressibility = 4.5e-5

; Electrostática y VdW
cutoff-scheme = Verlet
coulombtype   = PME
rcoulomb      = 1.0
rvdw          = 1.0
pbc           = xyz

; Restricciones
constraints   = h-bonds
constraint-algorithm = LINCS

; Sin restricciones de posición en producción
"""

# Guardar los archivos
import os
os.makedirs('mdp_files', exist_ok=True)
for nombre, contenido in [("01_em.mdp", mdp_minimizacion),
                           ("02_nvt.mdp", mdp_nvt),
                           ("03_npt.mdp", mdp_npt),
                           ("04_md.mdp",  mdp_produccion)]:
    with open(f"mdp_files/{nombre}", "w") as f:
        f.write(contenido)
    print(f"✓ Guardado: mdp_files/{nombre}")

## 4. Monitoreo de Equilibración

Es fundamental verificar que el sistema ha alcanzado equilibrio antes de la etapa de producción. Las propiedades a monitorear son:

| Propiedad | Etapa | Señal de equilibrio |
|-----------|-------|--------------------|
| **Temperatura** | NVT | Fluctuaciones estables alrededor de T objetivo |
| **Presión** | NPT | Fluctuaciones estables alrededor de P objetivo |
| **Densidad / Volumen** | NPT | Plateau estable |
| **Energía potencial** | EM + NVT | Convergencia (mínimo) |
| **RMSD proteína** | NVT/NPT | Plateau (sin drift) |

In [ ]:
# Simular curvas de equilibración típicas
np.random.seed(0)
n_ps = 1000  # 1 ns de simulación en pasos de 1 ps
t_ps = np.arange(n_ps)

# Temperatura: calentamiento NVT
T_equil = np.zeros(n_ps)
T_equil[:100] = np.linspace(0, 300, 100)  # calentamiento
T_equil[100:] = 300 + np.random.normal(0, 3, n_ps - 100)  # equilibrado

# Densidad: ajuste NPT
rho_start = 980  # kg/m³ (valor inicial)
rho_target = 997  # kg/m³ (agua a 300 K)
rho = rho_start + (rho_target - rho_start) * (1 - np.exp(-t_ps / 200))
rho += np.random.normal(0, 0.5, n_ps)

# Energía potencial: minimización
E_pot = -50000 * (1 - np.exp(-t_ps / 100)) + np.random.normal(0, 50, n_ps)

fig, axes = plt.subplots(3, 1, figsize=(12, 10))

axes[0].plot(t_ps, T_equil, 'b-', linewidth=1)
axes[0].axhline(300, color='r', linestyle='--', linewidth=1.5, label='T = 300 K')
axes[0].axvline(100, color='g', linestyle=':', linewidth=2, label='Fin calentamiento')
axes[0].set_ylabel('Temperatura (K)', fontsize=12)
axes[0].set_title('Monitoreo de Equilibración - Temperatura', fontsize=13)
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(t_ps, rho, 'g-', linewidth=1)
axes[1].axhline(rho_target, color='r', linestyle='--', linewidth=1.5, label=f'ρ = {rho_target} kg/m³')
axes[1].set_ylabel('Densidad (kg/m³)', fontsize=12)
axes[1].set_title('Densidad del Sistema', fontsize=13)
axes[1].legend(); axes[1].grid(True, alpha=0.3)

axes[2].plot(t_ps, E_pot, 'm-', linewidth=1)
axes[2].set_ylabel('Energía Potencial (kJ/mol)', fontsize=12)
axes[2].set_xlabel('Tiempo (ps)', fontsize=12)
axes[2].set_title('Energía Potencial del Sistema', fontsize=13)
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('equilibracion.png', dpi=100, bbox_inches='tight')
plt.show()

## 5. Simulación NVT Completa con OpenMM

A continuación se muestra cómo configurar una simulación NVT y NPT con OpenMM.

In [ ]:
# Simulación NVT con OpenMM (si está disponible)
try:
    import openmm as mm
    import openmm.app as app
    import openmm.unit as unit

    # --- Configuración del sistema ---
    # Cargar una topología PDB de ejemplo
    # pdb = app.PDBFile('mi_proteina_solvatada.pdb')
    # forcefield = app.ForceField('amber14-all.xml', 'amber14/tip3pfb.xml')
    # system = forcefield.createSystem(
    #     pdb.topology,
    #     nonbondedMethod=app.PME,
    #     nonbondedCutoff=1.0*unit.nanometer,
    #     constraints=app.HBonds
    # )

    # --- Integrador Langevin (NVT) ---
    # integrator = mm.LangevinMiddleIntegrator(
    #     300*unit.kelvin,     # temperatura
    #     1.0/unit.picosecond, # coeficiente de fricción
    #     0.002*unit.picosecond  # paso de tiempo
    # )

    # --- Barostato Monte Carlo para NPT ---
    # system.addForce(mm.MonteCarloBarostat(1*unit.bar, 300*unit.kelvin))

    # --- Plataforma de cómputo ---
    # platform = mm.Platform.getPlatformByName('CUDA')  # o 'CPU', 'OpenCL'

    # --- Simulación ---
    # simulation = app.Simulation(pdb.topology, system, integrator, platform)
    # simulation.context.setPositions(pdb.positions)

    # --- Minimización ---
    # simulation.minimizeEnergy(maxIterations=1000)

    # --- Registrar propiedades ---
    # simulation.reporters.append(
    #     app.StateDataReporter('md.log', 5000,
    #         step=True, time=True, potentialEnergy=True,
    #         temperature=True, density=True)
    # )
    # simulation.reporters.append(app.DCDReporter('traj.dcd', 5000))

    # --- Ejecutar ---
    # simulation.step(500000)  # 1 ns

    print("OpenMM disponible. Descomenta el código y proporciona tu archivo PDB solvatado.")

except ImportError:
    print("OpenMM no está instalado.")
    print("Para instalarlo: conda install -c conda-forge openmm")
    print()
    print("Mostrando ejemplo de código OpenMM (modo demostración)")

# Código equivalente en GROMACS (línea de comandos)
codigo_gromacs = """
# Flujo de trabajo GROMACS completo
# ==================================

# 1. Minimización de energía
gmx grompp -f mdp_files/01_em.mdp  -c sistema_solvatado.gro -p topol.top -o em.tpr
gmx mdrun  -v -deffnm em

# 2. Equilibración NVT (100 ps)
gmx grompp -f mdp_files/02_nvt.mdp -c em.gro  -r em.gro -p topol.top -o nvt.tpr
gmx mdrun  -v -deffnm nvt

# 3. Equilibración NPT (100 ps)
gmx grompp -f mdp_files/03_npt.mdp -c nvt.gro -r nvt.gro -t nvt.cpt -p topol.top -o npt.tpr
gmx mdrun  -v -deffnm npt

# 4. Producción MD (10 ns)
gmx grompp -f mdp_files/04_md.mdp  -c npt.gro -t npt.cpt -p topol.top -o md.tpr
gmx mdrun  -v -deffnm md
"""
print(codigo_gromacs)

## 6. Ejercicios

### Ejercicio 1 (Básico)
Modifica el archivo `02_nvt.mdp` para cambiar el termostato de `V-rescale` a `Nose-Hoover` y ajusta los parámetros correspondientes (`tau_t = 0.5 ps` para Nosé-Hoover es más típico).

### Ejercicio 2 (Intermedio)
Implementa el termostato de Nosé-Hoover simplificado en Python para un oscilador armónico. Verifica que la distribución de velocidades sigue a Maxwell-Boltzmann comparando con el termostato de Berendsen.

### Ejercicio 3 (Avanzado)
Usando OpenMM, configura una simulación NVT y NPT para una caja de agua (use `app.Modeller` con `addSolvent`). Monitorea temperatura, presión y densidad durante 100 ps. Determina el tiempo necesario para que el sistema alcance equilibrio.

In [ ]:
# Ejercicio 2 - Termostato Nosé-Hoover simplificado
def simular_nose_hoover(x0, v0, dt, n_pasos, k=1.0, m=1.0, T_obj=1.0, Q=1.0):
    """
    Integrador Nosé-Hoover para oscilador armónico 1D.
    Ecuaciones de movimiento extendidas.
    
    Args:
        x0, v0: posición y velocidad inicial
        dt: paso de tiempo
        n_pasos: número de pasos
        k, m: constante del resorte y masa
        T_obj: temperatura objetivo (en unidades reducidas, k_B=1)
        Q: masa del baño térmico (inercia del termostato)
    
    Returns:
        tiempos, posiciones, velocidades, temperaturas instantáneas
    """
    x, v, xi = x0, v0, 0.0  # xi: variable del baño de Nosé-Hoover
    posiciones, velocidades, temps = [x], [v], [m * v**2]

    for _ in range(1, n_pasos):
        F = -k * x
        # Integración orden 1 (simplificado para demostración)
        a = F / m - xi * v
        xi_dot = (m * v**2 - T_obj) / Q  # k_B = 1 en unidades reducidas
        x = x + v * dt
        v = v + a * dt
        xi = xi + xi_dot * dt

        posiciones.append(x)
        velocidades.append(v)
        temps.append(m * v**2)  # 2*Ek (1D, k_B=1)

    return np.array(posiciones), np.array(velocidades), np.array(temps)

pos_nh, vel_nh, T_nh = simular_nose_hoover(
    x0=1.0, v0=0.0, dt=0.01, n_pasos=100000,
    k=1.0, m=1.0, T_obj=1.0, Q=1.0
)

# Distribución de velocidades
from scipy.stats import norm
v_range = np.linspace(-4, 4, 200)
f_mb = norm.pdf(v_range, 0, 1)  # T=1, m=1, k_B=1

plt.figure(figsize=(8, 5))
plt.hist(vel_nh[5000:], bins=60, density=True, alpha=0.6, color='steelblue', label='Nosé-Hoover (simulado)')
plt.plot(v_range, f_mb, 'r-', linewidth=2.5, label='Maxwell-Boltzmann (T=1)')
plt.xlabel('Velocidad (u.r.)', fontsize=12)
plt.ylabel('Densidad de probabilidad', fontsize=12)
plt.title('Distribución de velocidades - Termostato Nosé-Hoover', fontsize=13)
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('nose_hoover_distribuccion.png', dpi=100, bbox_inches='tight')
plt.show()
print(f"T media (Nosé-Hoover) = {np.mean(T_nh[5000:])/2:.4f} (esperado: 0.5 en k_B=1)")

## 7. Recursos Adicionales

- **Documentación GROMACS:**
  - [Temperature Coupling](https://manual.gromacs.org/documentation/current/reference-manual/algorithms/temperature-coupling.html)
  - [Pressure Coupling](https://manual.gromacs.org/documentation/current/reference-manual/algorithms/pressure-coupling.html)
  - [mdp options reference](https://manual.gromacs.org/documentation/current/user-guide/mdp-options.html)

- **OpenMM:**
  - [OpenMM Integrators](https://openmm.org/documentation/latest/api-python/app.html#integrators)
  - [OpenMM User Guide](https://openmm.org/documentation/latest/userguide/)

- **Artículos clave:**
  - Bussi et al., *J. Chem. Phys.* 126, 014101 (2007) — v-rescale
  - Nosé, *Mol. Phys.* 52, 255 (1984) — Nosé-Hoover
  - Berendsen et al., *J. Chem. Phys.* 81, 3684 (1984) — Berendsen

---

## ✅ Verificación de Aprendizaje

Al finalizar esta actividad deberías ser capaz de:

- ✅ Configurar archivos `.mdp` de GROMACS para NVT y NPT
- ✅ Comparar los termostatos Berendsen, v-rescale y Nosé-Hoover
- ✅ Implementar el acoplamiento de temperatura en Python
- ✅ Monitorear la equilibración del sistema mediante gráficas de T, P y densidad
- ✅ Validar que el sistema ha alcanzado equilibrio antes de la producción

---

<div align="center">

## 🎉 ¡Felicitaciones!

Has completado la **Actividad 5.4: Termostatos y Barostatos**

[![Anterior](https://img.shields.io/badge/⬅️_Actividad_5.3-Condiciones_de_Contorno-blue.svg)](03_condiciones_contorno_ensemble.ipynb)
[![Siguiente](https://img.shields.io/badge/Actividad_5.5_➡️-Preparación_de_Sistemas-green.svg)](05_preparacion_sistemas.ipynb)

---

📚 **[Volver al Módulo 5](README.md)** | 🏠 **[Inicio del Curso](../README.md)**

---

**Universidad de Caldas - Departamento de Química**  
*Química Computacional 173G7G*

</div>